# Kognis Lite — Offline Humanitarian Field Assistant
### Gemma 4 Good Hackathon · Demo Notebook

This notebook demonstrates the **knowledge pipeline** that powers Kognis Lite:

1. Load humanitarian corpus (Sphere, WHO, ICRC, INSARAG guidelines)
2. Embed with `multilingual-e5-small` (384-dim, same model as on-device ONNX)
3. Build BM25 + HNSW retrieval index
4. Run a retrieval-augmented query against **Gemma 4 (gemma-3-4b-it)** via the Gemma API
5. Parse LOCATION_JSON tags → show waypoint on a folium map

**Track:** Google Global Resilience Challenge + Special Track: LiteRT on Device

> Runtime note: This notebook runs the cloud inference path (Gemma API). The actual
> app runs Gemma 4 E2B fully offline on Android via LiteRT-LM 0.11.0.
---

In [ ]:
%%capture
!pip install -q sentence-transformers rank_bm25 google-generativeai folium

In [ ]:
import json, re, math, textwrap
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import google.generativeai as genai
from IPython.display import display, Markdown

## 1 · Humanitarian Corpus

In production the app ingests `humanitarian_base.json` from assets (generated by
`pipeline/vectorize_corpus.py`). Here we use a representative seed for demonstration.

In [ ]:
# Seed corpus — bilingual humanitarian guidelines (EN/ES)
# Replace or extend with the full INSARAG / Sphere dataset via vectorize_corpus.py
CORPUS = [
    {
        "chunk_id": "sphere-wash-001",
        "title": "Sphere — WASH Minimum Standards",
        "content": "Minimum 15 litres of safe water per person per day for drinking, cooking and personal hygiene. Water point within 500m of shelter. Defecation facilities max 50 persons per latrine."
    },
    {
        "chunk_id": "who-triage-001",
        "title": "WHO — START Triage (Mass Casualty)",
        "content": "START triage: RED (immediate) life-saving intervention needed within minutes. YELLOW (delayed) serious but stable, 30min–2h. GREEN (minor) walking wounded. BLACK (deceased/expectant) no vital signs or unsurvivable given available resources."
    },
    {
        "chunk_id": "icrc-bleeding-001",
        "title": "ICRC — External Haemorrhage Control",
        "content": "1. Direct pressure with clean cloth ≥5 min without lifting. 2. For limb wounds not controlled by pressure: tourniquet 5–7 cm proximal to wound, never over a joint. Mark application time on patient. 3. Haemostatic dressing for junctional wounds (neck, groin, axilla) where tourniquet not feasible."
    },
    {
        "chunk_id": "insarag-search-001",
        "title": "INSARAG — Urban Search and Rescue Sectorisation",
        "content": "USAR operations divide the affected area into sectors. Each sector assigned to one team. Search follows LAST principle: Listen, Ask (survivors), Search (systematic), Treat and transport. Marking system: single slash = team searching; X = searched, number of live/dead found."
    },
    {
        "chunk_id": "ocha-coordination-001",
        "title": "OCHA — Cluster System Coordination",
        "content": "Humanitarian clusters: Health (WHO), WASH (UNICEF), Food (WFP/FAO), Shelter (UNHCR/IFRC), Nutrition (UNICEF), Protection (UNHCR), Education (UNICEF/Save), Logistics (WFP), ETC (WFP). Activate at field level via Humanitarian Coordinator."
    },
    {
        "chunk_id": "msf-medevac-001",
        "title": "MSF — Medical Evacuation Request",
        "content": "MEDEVAC 9-Line: 1) Location (grid/coords), 2) Radio freq, 3) Patient count by precedence (urgent/priority/routine), 4) Special equipment (hoist/jungle penetrator), 5) Patient count by nationality, 6) Security, 7) Method of marking LZ, 8) Patient nationality/status, 9) CBRN hazard."
    },
    {
        "chunk_id": "sphere-nutrition-001",
        "title": "Sphere — Nutrition Minimum Standards",
        "content": "Minimum dietary energy requirement: 2,100 kcal/person/day in acute emergency. MUAC <11.5 cm = severe acute malnutrition in children 6–59 months. Therapeutic feeding (RUTF) for SAM; supplementary feeding (RUSF) for MAM."
    },
    {
        "chunk_id": "who-cholera-001",
        "title": "WHO — Cholera Case Management",
        "content": "Severe cholera: IV Ringer's Lactate 100 mL/kg in 3h (adults). Oral rehydration salts for mild/moderate. Antibiotic (doxycycline 300 mg single dose adult) reduces duration. Isolate patients; 0.05% chlorine for environmental decontamination."
    },
]

print(f"Corpus: {len(CORPUS)} chunks loaded")

## 2 · Embed with multilingual-e5-small

Same model as the ONNX runtime on device → identical semantic space.

In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

texts = [f"passage: {c['title']}. {c['content']}" for c in CORPUS]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")

## 3 · Hybrid Retrieval (BM25 + Cosine → RRF)

Mirrors the on-device pipeline: BM25 keyword scores + HNSW cosine similarity fused with
Reciprocal Rank Fusion (k=60).

In [ ]:
# BM25 index
tokenized = [c['content'].lower().split() for c in CORPUS]
bm25 = BM25Okapi(tokenized)

def hybrid_retrieve(query: str, top_k: int = 3, rrf_k: int = 60) -> list[dict]:
    # BM25
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1]

    # Dense (cosine)
    q_emb = model.encode([f"query: {query}"], normalize_embeddings=True)[0]
    cos_scores = embeddings @ q_emb
    cos_ranked = np.argsort(cos_scores)[::-1]

    # RRF fusion
    rrf = {}
    for rank, idx in enumerate(bm25_ranked):
        rrf[idx] = rrf.get(idx, 0) + 1 / (rrf_k + rank + 1)
    for rank, idx in enumerate(cos_ranked):
        rrf[idx] = rrf.get(idx, 0) + 1 / (rrf_k + rank + 1)

    top = sorted(rrf, key=lambda i: rrf[i], reverse=True)[:top_k]
    return [CORPUS[i] for i in top]

# Quick sanity check
hits = hybrid_retrieve("tourniquet bleeding control")
print("Test retrieval:")
for h in hits:
    print(f"  · {h['title']}")

## 4 · RAG Query via Gemma 4

Set your Gemma API key below (Google AI Studio → `makersuite.google.com`).

In [ ]:
import os
# Set GEMMA_API_KEY in Colab Secrets (key icon on left sidebar)
# or set it directly here for testing:
# os.environ['GEMMA_API_KEY'] = 'YOUR_KEY_HERE'

try:
    from google.colab import userdata
    GEMMA_API_KEY = userdata.get('GEMMA_API_KEY')
except Exception:
    GEMMA_API_KEY = os.environ.get('GEMMA_API_KEY', '')

if not GEMMA_API_KEY:
    print("⚠️  No API key found. Section 4 will be skipped.")
    print("   Add GEMMA_API_KEY to Colab Secrets or set os.environ['GEMMA_API_KEY'].")
else:
    print("✅  API key loaded.")
    genai.configure(api_key=GEMMA_API_KEY)

In [ ]:
SYSTEM_PROMPT = """You are Kognis Lite, a humanitarian field assistant.
Rules:
- Use only the CONTEXT below. Cite sources briefly.
- If a location is mentioned, append exactly one LOCATION_JSON on its own line:
  LOCATION_JSON: {\"lat\": <float>, \"lon\": <float>, \"label\": \"<name>\"}
- If you don't have the information: \"I don't have that information in my humanitarian knowledge base.\"
- Be concise. Field conditions, not academic style."""

def rag_query(user_query: str) -> str:
    if not GEMMA_API_KEY:
        return "[API key not set — skipping Gemma inference]"

    chunks = hybrid_retrieve(user_query)
    context_block = "\n\n".join(
        f"[Source: {c['title']}]\n{c['content']}" for c in chunks
    )

    full_prompt = f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{context_block}\n\nQUESTION: {user_query}"

    gemma = genai.GenerativeModel("gemma-3-4b-it")
    resp = gemma.generate_content(full_prompt)
    return resp.text

# Demo queries
QUERIES = [
    "How do I control severe bleeding from a limb wound?",
    "What is the minimum water requirement per person per day in an emergency?",
    "Describe the START triage categories for mass casualty incidents.",
]

for q in QUERIES:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print("="*60)
    answer = rag_query(q)
    display(Markdown(answer))

## 5 · Map Visualisation

If the model emitted a `LOCATION_JSON` tag, parse it and render on a folium map.
On-device this drives the OsmAnd/osmdroid map overlay.

In [ ]:
import folium

# LOCATION_JSON now carries an optional 'type' field (S17 / F11):
# LOCATION_JSON: {"lat": 18.44, "lon": -69.94, "label": "Medical Post", "type": "MEDICAL"}
# The regex uses [^}]+ which already handles the extra field; json.loads handles it too.

SAR_MARKER_COLORS = {
    "VICTIM":     "red",
    "MEDICAL":    "darkred",
    "HAZARD":     "orange",
    "COMMAND":    "blue",
    "EXTRACTION": "green",
    "MISSING":    "orange",
    "BASE":       "beige",
    "WATER":      "lightblue",
}

def extract_location(text: str):
    """Parse LOCATION_JSON from model output, tolerating optional 'type' field."""
    m = re.search(r'LOCATION_JSON:\s*(\{[^}]+\})', text)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None

# Run a location-aware query
location_query = "There's a mass casualty incident near the Mogadishu port (2.0469° N, 45.3182° E). What is the triage protocol?"
loc_answer = rag_query(location_query)
display(Markdown(f"**Query:** {location_query}\n\n{loc_answer}"))

loc = extract_location(loc_answer)
if loc:
    marker_type = loc.get('type', 'COMMAND')
    color = SAR_MARKER_COLORS.get(marker_type, 'blue')
    m = folium.Map(location=[loc['lat'], loc['lon']], zoom_start=13)
    folium.Marker(
        [loc['lat'], loc['lon']],
        popup=f"{loc.get('label', 'Kognis marker')} [{marker_type}]",
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(m)
    display(m)
else:
    print("No LOCATION_JSON found in response (API key may not be set).")


## 6 · SAR Marker Types (INSARAG-aligned)

Session S17 (F10) replaced the 6 military CoT types with 8 INSARAG SAR types.
On-device each marker is rendered as a **coloured circle with a letter/symbol** drawn via Canvas.
The LLM selects the type via the optional `"type"` field in `LOCATION_JSON` (F11).


In [ ]:
SAR_TYPES = [
    {"type": "VICTIM",     "symbol": "V",  "color": "red",       "use_case": "Known victim location — injured or trapped"},
    {"type": "MEDICAL",    "symbol": "+",  "color": "dark red",   "use_case": "Medical post, triage point, first-aid station"},
    {"type": "HAZARD",     "symbol": "!",  "color": "amber",      "use_case": "Hazardous material, structural collapse risk, toxic area"},
    {"type": "COMMAND",    "symbol": "C",  "color": "blue",       "use_case": "Command post, coordination point (default fallback)"},
    {"type": "EXTRACTION", "symbol": "X",  "color": "green",      "use_case": "Extraction / rescue in progress or completed"},
    {"type": "MISSING",    "symbol": "M",  "color": "orange",     "use_case": "Last known position of missing person"},
    {"type": "BASE",       "symbol": "B",  "color": "brown",      "use_case": "Base of operations, staging area, logistics hub"},
    {"type": "WATER",      "symbol": "W",  "color": "light blue", "use_case": "Water source, distribution point, WASH station"},
]

print(f"{'Type':<12} {'Symbol':<8} {'Color':<12} Use case")
print("-" * 72)
for t in SAR_TYPES:
    print(f"{t['type']:<12} {t['symbol']:<8} {t['color']:<12} {t['use_case']}")

print()
print("LOCATION_JSON example with type field:")
example = {"lat": 18.44850, "lon": -69.94177, "label": "Medical Post Alpha", "type": "MEDICAL"}
print(f'  LOCATION_JSON: {json.dumps(example)}')
print()
print("Fallback: if 'type' absent or unrecognised, MapOptionSheet defaults to COMMAND.")


## 7 · Embedding Prefix Requirement (multilingual-e5-small)

`intfloat/multilingual-e5-small` is an **asymmetric** model that requires different
prefixes for indexed passages vs. search queries:

| Role | Prefix | Example |
|------|--------|---------|
| Corpus document (indexed) | `passage: ` | `passage: Sphere — WASH. Minimum 15 L...` |
| Search query (at inference) | `query: ` | `query: water requirement per person` |

**Without the correct prefix the cosine similarity scores are degraded**, especially for
cross-lingual queries (EN query vs. ES corpus). F18 added the `"query: "` prefix inside
`EmbeddingEngine.kt` so all device-side queries benefit automatically.

The corpus cells in this notebook already use `"passage: "` (see Section 2). The retrieval
cell already uses `"query: "` in the dense-search path.


## 8 · Eval Results

F12 added an in-app automated eval runner (gear menu → "Run RAG Eval"). It processes
`sar_test.json` — 50 questions through the full RAG+LLM pipeline — and exports:

```json
{
  "eval_date": "20260516_143021",
  "kb_version": 6,
  "total_questions": 50,
  "timed_out_count": 0,
  "multi_chunk_count": 30,
  "hallucination_count": 20,
  "results": [
    {
      "id": "mc-001",
      "category": "multi_chunk",
      "question": "What is the minimum water requirement per person per day?",
      "answer": "15 litres of safe water per person per day (Sphere WASH).",
      "duration_ms": 3241,
      "timed_out": false
    }
  ]
}
```

`clearConversation()` (F13) is called before each question to reset KV-cache and prevent
native OOM crashes during back-to-back inference.


In [ ]:
# Load and summarise an eval export
# Swap the path below to your own kognis_eval_YYYYMMDD_HHmmss.json file.

import os, pathlib

EVAL_JSON_PATH = None  # set to your file path, e.g. "/sdcard/kognis_eval_20260516_143021.json"

if EVAL_JSON_PATH and pathlib.Path(EVAL_JSON_PATH).exists():
    with open(EVAL_JSON_PATH) as f:
        eval_data = json.load(f)

    results   = eval_data["results"]
    total     = eval_data["total_questions"]
    timed_out = eval_data["timed_out_count"]
    mc_count  = eval_data["multi_chunk_count"]
    hal_count = eval_data["hallucination_count"]
    durations = [r["duration_ms"] for r in results if not r["timed_out"]]

    print(f"Eval date    : {eval_data['eval_date']}")
    print(f"KB version   : {eval_data['kb_version']}")
    print(f"Total Qs     : {total}  (multi-chunk: {mc_count}, hallucination: {hal_count})")
    print(f"Timed out    : {timed_out} ({timed_out/total*100:.1f}%)")
    print(f"Avg latency  : {sum(durations)/len(durations)/1000:.2f}s per question")
    print(f"Max latency  : {max(durations)/1000:.2f}s")
else:
    print("[No eval file loaded — showing S17 reference results]")

print()
print("Reference results from S17 stress test (50 questions on S24 Ultra):")
ref = [
    ("Total questions",          "50 (30 multi-chunk + 20 hallucination probes)"),
    ("Timed out",                "0  (0%)"),
    ("RAG hit rate",             "0.96  (96% of questions used ≥1 RAG chunk)"),
    ("Hallucination resistance", "100%  (0/20 probes triggered false facts)"),
    ("Avg TPS (cold)",           "~21 tok/s"),
    ("Avg TPS (sustained)",      "~14 tok/s  (thermal throttle after ~15 questions)"),
    ("Peak temperature",         "75.9°C  (S24 Ultra, Snapdragon 8 Elite)"),
]
for label, value in ref:
    print(f"  {label:<28}: {value}")


## 9 · Performance Metrics Summary


In [ ]:
METRICS = {
    "rag_hit_rate":            0.96,
    "hallucination_resistance": 1.00,   # 100% — 0/20 probes triggered
    "avg_tps_cold":            21.0,    # tok/s, first ~15 questions
    "avg_tps_sustained":       14.1,    # tok/s, after thermal throttle onset
    "thermal_note":            "75.9°C peak during 50-question stress test on S24 Ultra",
    "kb_chunks":               1153,
    "kb_version":              6,
    "embedding_model":         "intfloat/multilingual-e5-small (384-dim, ONNX on-device)",
    "llm":                     "Gemma 4 E2B (LiteRT-LM 0.11.0, GPU / Adreno 750)",
    "eval_questions":          50,
    "nnapi_note":              "ONNX NNAPI delegate enabled; NPU compute-unit pinning pending — important for field thermal management",
}

print("=" * 60)
print("Kognis Lite — S17 Performance Metrics")
print("=" * 60)
for k, v in METRICS.items():
    label = k.replace('_', ' ').title()
    if isinstance(v, float) and v <= 1.0 and 'tps' not in k:
        print(f"  {label:<30}: {v*100:.0f}%")
    else:
        print(f"  {label:<30}: {v}")
print("=" * 60)
print()
print("Thermal note:", METRICS['thermal_note'])
print("NNAPI note  :", METRICS['nnapi_note'])


## 6 · Vectorise Your Own Corpus

To build the full on-device knowledge base:

```bash
# 1. Place source documents in pipeline/docs/
# 2. Run the vectoriser
cd pipeline
python vectorize_corpus.py --input docs/ --output ../app/src/main/assets/humanitarian_base.json

# Output: [{"title": ..., "content": ..., "chunk_id": ..., "vector": [384 floats]}]
# 3. Build & deploy the app — KnowledgeBaseLoader.ingestIfNeeded() picks it up on first launch
```

The vectoriser uses the same `intfloat/multilingual-e5-small` model, so embeddings are
directly compatible with the on-device ONNX runtime.
---

## Architecture Summary

```
┌──────────────────────────────────────────────────────┐
│                  Android Device (offline)             │
│                                                      │
│  User query                                          │
│       │                                              │
│       ▼                                              │
│  RagOrchestrator                                     │
│   ├─ EmbeddingEngine (ONNX e5-small)                │
│   ├─ ObjectBox HNSW  → top-K dense hits             │
│   ├─ BM25Searcher    → top-K keyword hits           │
│   └─ RRF fusion → context block                     │
│       │                                              │
│       ▼                                              │
│  FieldAssistantService                              │
│   └─ LiteRtModelRunner (Gemma 4 E2B, GPU)          │
│       │                                              │
│       ▼                                              │
│  MainActivity (Compose UI)                          │
│   ├─ Token streaming                                │
│   ├─ LOCATION_JSON → OsmAndBridge / osmdroid        │
│   └─ GPX export → OsmAnd / share                   │
└──────────────────────────────────────────────────────┘
```
**Zero network I/O** — model, embeddings, and knowledge base all local.